In [1]:
import numpy as np
import stdpopsim
import random
from typing import List, Dict, Optional

def sampling_populations(model):
    populations = []
    for pop in model.populations:
        if hasattr(pop, 'default_sampling_time'):
            if isinstance(pop.default_sampling_time, float):
                if pop.default_sampling_time > 0:
                    pass
            elif pop.allow_samples:
                populations.append(pop)
    return populations

def is_any_numeric_or_roman_numeral(item):
    # includes C. elegans chromosomes except X
    for char in item:
        if char.isdigit() or char in ['I', 'II', 'III', 'IV', 'V', ]: # removes X
            if char == 'CM009947.2':
                return False
            else: return True#False
    if item == 'Mt' or item == 'Pt':
        return False
    return False

def random_sample_counts(
    sampling_populations: List[int], num_samples: int = 25, seed: Optional[int] = None
    ) -> Dict[str, int]:
    """
    Randomly distributes `n` samples across the given populations.
    """
    rng = random.Random(seed)
    sampled_counts = {pop.name: 0 for pop in sampling_populations}  
    for pop in rng.choices(sampling_populations, k=num_samples):
        sampled_counts[pop.name] += 1  
    return sampled_counts

def sample_chromosome(species):
    chromosomes = [
        chrom for chrom in species.genome.chromosomes
        if is_any_numeric_or_roman_numeral(chrom.id)
    ]
    chromosome = species.genome.chromosomes[np.random.randint(0, len(chromosomes))]
    while chromosome.id in ['Mt', 'Pt']:
        chromosome = species.genome.chromosomes[np.random.randint(0, len(chromosomes))]
    return chromosome


/home/kkor/miniconda/envs/cxt/lib/python3.12/site-packages/stdpopsim/catalog/HomSap/demographic_models.py:158: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  time=int(extended_GF.time.head(1) - 1), rate=0
/home/kkor/miniconda/envs/cxt/lib/python3.12/site-packages/stdpopsim/catalog/HomSap/demographic_models.py:161: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  time=int(extended_GF.time.tail(1) + 1), rate=0


In [2]:
def simulate_random_segment(
    seed,
    num_samples=25,
    segment_length=1e6, 
    species_name="HomSap",
    genetic_map=None,
    population_size=None
):

    np.random.seed(seed)
    seed = np.random.randint(1, 2**32)
    species = stdpopsim.get_species(species_name)

    demographic_models = species.demographic_models
    if len(demographic_models) > 0:
        # Filter out models with non-present sampling points
        excluded_descriptions = {
            'Multi-population model of ancient Eurasia',
            'Out-of-Africa with archaic admixture into Papuans',
            'Multi-population model of ancient Europe'
        }
        valid_models = [model for model in demographic_models 
                        if model.description not in excluded_descriptions]
    else:
        valid_models = [stdpopsim.PiecewiseConstantSize(population_size)]

    demography = np.random.choice(valid_models)
    populations = sampling_populations(demography)
    samples = random_sample_counts(populations, num_samples=num_samples, seed=seed)

    engine = stdpopsim.get_engine("msprime")

    if genetic_map is None:
        chromosome = sample_chromosome(species)
        left = np.random.uniform(chromosome.length - segment_length)
        right = left + segment_length
        contig = species.get_contig(
            chromosome.id, left=left, right=right, 
            mutation_rate=demography.mutation_rate
        )
    else:
        while True:
            chromosome = sample_chromosome(species)
            left  = np.random.uniform(0, chromosome.length - segment_length)
            right = left + segment_length
            try:
                contig = species.get_contig(
                    chromosome.id, left=left, right=right,
                    mutation_rate=demography.mutation_rate, genetic_map=genetic_map
                )
            except ValueError:           # "All intervals are missing data"
                continue                 # try another window
            if np.isfinite(contig.recombination_map.rate).all():
                break

    ts = engine.simulate(demography, contig, samples, seed=seed).trim()
    return ts



genetic_map = 'Campbell2016_CanFam3_1'
genetic_map = None
species_name = "CanFam"
population_size = 13_700
num_samples = 25
seed = 1224
segment_length = 1e6
species = stdpopsim.get_species(species_name)

In [ ]:
import os, sys
from functools import partial
from multiprocessing import get_context
from tqdm import tqdm
from cxt.utils import simulate_parameterized_tree_sequence

def dump_one_proc(i, outdir, sim_func):
    ts = sim_func(i)                  # top-level & picklable
    path = os.path.join(outdir, f"ts_{i:06d}.trees")
    ts.dump(path)
    return path

def generate_tree_sequences(num_samples, output_dir, ts_simulation_func, num_processes=8):
    os.makedirs(output_dir, exist_ok=True)
    worker = partial(dump_one_proc, outdir=output_dir, sim_func=ts_simulation_func)
    start_method = "fork"
    with get_context(start_method).Pool(num_processes) as pool:
        for _ in tqdm(pool.imap_unordered(worker, range(num_samples)),
                      total=num_samples, desc="Simulating"):
            pass

# example:
n_individuals = 25
simulate_parameterized_tree_sequence = partial(
    simulate_parameterized_tree_sequence, samples=n_individuals)

generate_tree_sequences(
    num_samples=1000, output_dir="ts_out",
    ts_simulation_func=simulate_parameterized_tree_sequence, num_processes=8)


Simulating:   2%|▎         | 25/1000 [00:00<00:08, 114.22it/s]

Simulating: 100%|██████████| 1000/1000 [00:07<00:00, 137.51it/s]


In [8]:
from cxt.utils import ts2X_vectorized
from cxt.utils import xor, xnor

In [9]:
import tskit
ts = tskit.load("ts_out/ts_000001.trees")

In [ ]:
from cxt.utils import retrieve_site_positions, calculate_window_sfs_vectorized
def ts2X_vectorized_bichan(ts, window_size=4000, step_size=2000,
                           pivot_A=0, pivot_B=1, offset=0):
    site_positions = retrieve_site_positions(ts) - offset
    gm = ts.genotype_matrix().T  # [samples, sites]

    # mask invariants / bad sites once
    bad = (gm >= 2).any(0) | (gm.sum(0) >= ts.num_samples)
    gm = gm[:, ~bad]
    site_positions = site_positions[~bad]

    num_samples = gm.shape[0]
    freq = gm.sum(0)                          # [sites]
    xor_mask = xor(gm[pivot_A], gm[pivot_B])  # [sites]  (0/1)

    w_xor  = freq * xor_mask
    w_xnor = freq * (1 - xor_mask)

    seq_len = ts.sequence_length  # avoid hardcoding 1e6
    w_multipliers = np.array([2, 8, 32, 64])
    n_steps = int(np.ceil(seq_len / step_size))

    # out: [2 channels, n_multipliers, n_steps, num_samples]
    Xs = np.zeros((2, len(w_multipliers), n_steps, num_samples), dtype=np.int32)

    for i, m in enumerate(w_multipliers):
        ws = window_size * m
        Xs[0, i] = calculate_window_sfs_vectorized(
            site_positions=site_positions,
            pivot_frequencies=w_xor,
            window_size=ws, step_size=step_size,
            sequence_length=seq_len, num_samples=num_samples
        )
        Xs[1, i] = calculate_window_sfs_vectorized(
            site_positions=site_positions,
            pivot_frequencies=w_xnor,
            window_size=ws, step_size=step_size,
            sequence_length=seq_len, num_samples=num_samples
        )

    return Xs  # cast to float16 later if needed

def process_X(ts, pairs, window_size=2000, dtype=np.float16):
    P = len(pairs)
    # probe shape once
    a0, b0 = pairs[0]
    X0 = ts2X_vectorized_bichan(ts, window_size=window_size, pivot_A=a0, pivot_B=b0)
    out = np.empty((P,) + X0.shape, dtype=X0.dtype)
    out[0] = X0
    for k, (pa, pb) in enumerate(pairs[1:], start=1):
        out[k] = ts2X_vectorized_bichan(ts, window_size=window_size, pivot_A=pa, pivot_B=pb)
    return out.astype(dtype, copy=False)


import numpy as np
import tskit
from typing import List, Tuple, Optional

def process_y(
    ts,
    pairs,
    window_size=2000,
    transform=None,
    dtype=np.float16,
    interp_fn=None,
    **interp_kwargs,
):
    """
    Compute window-averaged TMRCA for many pairs using YOUR interpolate function.
    y : (P, L) array
    """
    if interp_fn is None:
        interp_fn = interpolate_tmrcas

    P = len(pairs)
    if P == 0:
        return np.empty((0, 0), dtype=dtype)

    # Probe once to get L
    a0, b0 = pairs[0]
    y0 = np.asarray(interp_fn(ts, window_size=window_size, sample_a=a0, sample_b=b0, **interp_kwargs))
    if y0.ndim != 1:
        raise ValueError(f"interp_fn must return a 1D array; got shape {y0.shape}")
    L = y0.shape[0]

    out = np.empty((P, L), dtype=y0.dtype)
    out[0] = y0

    # Fill the rest
    for k, (pa, pb) in enumerate(pairs[1:], start=1):
        yk = np.asarray(interp_fn(ts, window_size=window_size, sample_a=pa, sample_b=pb, **interp_kwargs))
        if yk.shape[0] != L:
            raise ValueError(
                f"Inconsistent window count from interp_fn: pair 0 -> {L}, pair {k} -> {yk.shape[0]}.\n"
                "Hint: ensure (end-start) and window_size are identical across calls (e.g., pass the same "
                "sequence_length/start/end via **interp_kwargs or snap down to whole windows inside interp_fn)."
            )
        out[k] = yk

    if transform:
        out = np.log(out)
    return out.astype(dtype, copy=False)


def interpolate_tmrca_per_window_spanavg(
    lefts: np.ndarray,
    rights: np.ndarray,
    values: np.ndarray,
    *,
    interval_start: int = 0,
    interval_end: Optional[int] = None,
    interval_size: int = 2000,
) -> np.ndarray:
    """
    Exact length-weighted averages of a piecewise-constant signal (values over [lefts, rights))
    into fixed windows [interval_start + k*interval_size, ...).
    """
    assert lefts.ndim == rights.ndim == values.ndim == 1
    assert len(lefts) == len(rights) == len(values)
    assert np.all(rights[1:] >= rights[:-1]) and np.all(lefts[1:] >= lefts[:-1])

    if interval_end is None:
        interval_end = int(rights[-1])

    # Build window edges
    starts = np.arange(interval_start, interval_end, interval_size, dtype=np.int64)
    ends   = starts + interval_size
    nW = len(starts)

    numer = np.zeros(nW, dtype=np.float64)  # overlap * value
    # Two-pointer sweep over segments and windows: O(S + W)
    i = 0  # segment index
    j = 0  # window index
    S = len(lefts)

    while i < S and j < nW:
        a = max(lefts[i], starts[j])
        b = min(rights[i], ends[j])
        if b > a:
            numer[j] += (b - a) * values[i]
        # advance the pointer that ends first
        if rights[i] <= ends[j]:
            i += 1
        else:
            j += 1

    return numer / interval_size


def interpolate_tmrcas(
    ts: tskit.TreeSequence,
    window_size: int,
    sequence_length: Optional[int] = None,
    sample_a: int = 0,
    sample_b: int = 1,
) -> np.ndarray:
    """
    Exact windowed averages of TMRCA for a given pair of samples across a tree sequence.
    """
    if sequence_length is None:
        sequence_length = int(ts.sequence_length)

    lefts, rights, tmrcas = [], [], []
    for tree in ts.trees():
        left, right = tree.interval
        # TMRCA for the specified pair
        m = tree.mrca(sample_a, sample_b)
        tmrca = tree.time(m)
        lefts.append(left)
        rights.append(right)
        tmrcas.append(tmrca)

    lefts  = np.asarray(lefts, dtype=np.int64)
    rights = np.asarray(rights, dtype=np.int64)
    vals   = np.asarray(tmrcas, dtype=np.float64)

    return interpolate_tmrca_per_window_spanavg(
        lefts, rights, vals,
        interval_start=0,
        interval_end=sequence_length,
        interval_size=window_size,
    )


In [ ]:



num_pairs = 200
pairs = np.array([(i, j) for i in range(2 * n_individuals) for j in range(i + 1, 2 * n_individuals)])
pairs = pairs[np.random.choice(len(pairs), size=num_pairs, replace=False)]
X = process_X(ts, pairs)
y = process_y(ts, pairs, transform=np.log)